<a href="https://colab.research.google.com/github/chanu-24/self-driving-Car-Vision-/blob/main/VehicleToCameraTransform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import torch
import torch.nn as nn

class VehicleToCameraTransform(nn.Module):
    def __init__(self):
        super(VehicleToCameraTransform, self).__init__()

    def forward(self, points_ego, rot_ego_to_cam, trans_ego_to_cam):
        """
        Args:
            points_ego (Tensor): [B, N, 3] - 차량 기준 3D 좌표 점들
            rot_ego_to_cam (Tensor): [B, Num_Cam, 3, 3] - 회전 행렬
            trans_ego_to_cam (Tensor): [B, Num_Cam, 3] - 이동 벡터
        Returns:
            points_cam (Tensor): [B, Num_Cam, N, 3] - 6대 카메라 기준으로 변환된 3D 좌표 점들
        """
        B, N, _ = points_ego.shape
        _, Num_Cam, _, _ = rot_ego_to_cam.shape

        # 1. 이동 벡터(trans) 연산을 위해 points_ego와 차원을 맞추기 위해 6대 카메라 축(Num_Cam)을 확장합니다.
        # points_ego를 [B, 1, N, 3] -> [B, Num_Cam, N, 3] 으로 복사/확장하세요.
        points_ego_expanded = points_ego.unsqueeze(1).expand(B, Num_Cam, N, 3)

        # 2. trans_ego_to_cam 역시 [B, Num_Cam, 1, 3]으로 차원을 조절합니다.
        trans_expanded = trans_ego_to_cam.unsqueeze(2) # [B, Num_Cam, 1, 3]

        # 3. [수식 반영]: P_ego - T_cam_to_ego 단계를 먼저 수행합니다.
        points_translated = points_ego_expanded + trans_expanded # 힌트: 구조에 따라 더하거나 빼기 진행

        # 4. 회전 행렬(Rot)과의 행렬 곱 연산을 수행합니다.
        # [B, Num_Cam, 3, 3] 행렬과 [B, Num_Cam, N, 3] 벡터 그룹을 효율적으로 연산하기 위해 torch.einsum을 활용합니다.
        # 행렬 수식: P_cam = R * P_translated
        # 빈칸인 einsum 수식 문자열을 완성해 보세요.
        points_cam = torch.einsum('bcij, bcnj -> bcni', rot_ego_to_cam, points_translated)

        return points_cam

# 테스트용 더미 텐서 생성
B, Num_Cam, N = 2, 6, 100
points_ego = torch.randn(B, N, 3)
rot_ego_to_cam = torch.randn(B, Num_Cam, 3, 3)
trans_ego_to_cam = torch.randn(B, Num_Cam, 3)

transform = VehicleToCameraTransform()
output = transform(points_ego, rot_ego_to_cam, trans_ego_to_cam)
print("최종 변환된 출력 텐서 구조:", output.shape) # 기대값: [2, 6, 100, 3]



최종 변환된 출력 텐서 구조: torch.Size([2, 6, 100, 3])
